<a href="https://colab.research.google.com/github/n5n0/eskwelabs-portfolio/blob/main/py7_tables_practice_bank_Crinan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# py7 · Tables: Python Practice Bank (Day 7)

**DABC Sprint 2 · AML case · the same five tables you queried in SQL on Days 1–4**

### How to use this bank

- Questions are tiered **EASY · AVERAGE · DIFFICULT**. Work in order.
- **In class:** Practice block 1 (18:50) → **Q1–10**, after the account behaviour table. Practice block 2 (19:50) → **Q11–25**. Your mentor will tell you the floor (what everyone finishes) for your room.
- **SQL twin** names the Day 1–4 question that asks the same thing. When both use the same table, your Python answer should give the same number as your SQL answer. If it doesn't, find out why: that's the lesson.
- Try each question yourself first. Then use AI to explain or check, following the sprint's AI protocol: paste the column names, never rows of data, and note it in your AI use log.
- ✍️ cells ask for a sentence, not code. Write it for the Head of AML.
- Your mentor has the answer key. Remaining questions are homework.

**Data you need** (Sprint 2 student folder, the same tables you queried in BigQuery on Days 1–4): `customers.csv`, `accounts.csv`, `transactions.csv`, `alerts.csv`, `investigations.csv` and `aml_analytical_extract.csv` (the Day 4 extract). Upload them to this Colab session, or put them in a Drive folder and add that folder to `DATA_DIRS` below.

## Step 0 · Setup

Run these four cells in order with **Shift + Enter**. After a runtime restart, run them again.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("pandas", pd.__version__, "· ready")

pandas 2.2.3 · ready


In [2]:
# ---------------------------------------------------------------------------
# CONFIGURATION — the only cell you should need to edit
# ---------------------------------------------------------------------------
USE_BIGQUERY = False                  # True = read the tables straight from BigQuery
PROJECT_ID   = "your-project-id"      # ← paste the project ID from your instructor
DATASET      = "aml_sprint2"

# Folders searched, in order, for the CSV files. Add your own Drive folder if needed.
DATA_DIRS = [
    ".",                                          # same folder as this notebook
    "data",
    "/content",                                   # Colab: files uploaded to this session
    "/content/drive/MyDrive/DABC24/aml_sprint2",  # Colab + Drive (mount Drive first)
]
print("USE_BIGQUERY =", USE_BIGQUERY)

USE_BIGQUERY = False


In [3]:
def load(name):
    """Return one table as a DataFrame.

    name is one of: customers, accounts, transactions, alerts, investigations,
    aml_analytical_extract. Prints where the data came from, so you always know.
    """
    if USE_BIGQUERY:
        from google.colab import auth
        from google.cloud import bigquery
        auth.authenticate_user()
        client = bigquery.Client(project=PROJECT_ID)
        frame = client.query(
            f"SELECT * FROM `{PROJECT_ID}.{DATASET}.{name}`"
        ).to_dataframe()
        print(f"{name:<24} {len(frame):>7,} rows  (BigQuery)")
        return frame

    for folder in DATA_DIRS:
        path = os.path.join(folder, f"{name}.csv")
        if os.path.exists(path):
            frame = pd.read_csv(path)
            print(f"{name:<24} {len(frame):>7,} rows  ({path})")
            return frame

    raise FileNotFoundError(
        f"{name}.csv not found in {DATA_DIRS}. Upload it to this session "
        "or add its folder to DATA_DIRS in the configuration cell."
    )

In [4]:
customers      = load("customers")
accounts       = load("accounts")
transactions   = load("transactions")
alerts         = load("alerts")
investigations = load("investigations")
df             = load("aml_analytical_extract")

customers                 10,000 rows  (./customers.csv)
accounts                  12,028 rows  (./accounts.csv)
transactions              46,838 rows  (./transactions.csv)
alerts                     2,618 rows  (./alerts.csv)
investigations             1,623 rows  (./investigations.csv)
aml_analytical_extract    46,793 rows  (./aml_analytical_extract.csv)


## Setup · standard cleaning

Run this once. It gives you `df` with `channel_clean` and `occupation_clean`, the de-duplicated `accounts_d` and `transactions_d`, and `alerts["alert_score_num"]`.

In [5]:
# Standard Day 6 cleaning, provided so this bank runs on its own.
# It uses the model decisions from the Day 6 cleaning log.
CHANNEL_MAP = {"atm": "ATM", "branch": "Branch", "instapay": "InstaPay",
               "mobile": "Mobile", "online": "Online", "pesonet": "PESONet"}
OCCUPATION_MAP = {"professional": "Professional", "business owner": "Business Owner",
                  "self-employed": "Business Owner", "student": "Student",
                  "unemployed": "Unemployed", "not employed": "Unemployed"}

df = df.copy()
df["channel_clean"] = df["channel"].str.strip().str.lower().map(CHANNEL_MAP)
df["occupation_clean"] = df["occupation"].str.strip().str.lower().map(OCCUPATION_MAP)

# De-duplicated source tables, same rule as 04_final_analytical_extract.sql
accounts_d = accounts.drop_duplicates(subset="acct_id", keep="first")
transactions_d = (transactions
                  .sort_values("tx_datetime", kind="stable")
                  .drop_duplicates(subset="tx_id", keep="first"))

# Numeric alert score: 'NOT_SCORED' becomes NaN
alerts["alert_score_num"] = pd.to_numeric(alerts["alert_score"], errors="coerce")

print(f"extract: {len(df):,} rows · accounts_d: {len(accounts_d):,} · "
      f"transactions_d: {len(transactions_d):,} · alerts: {len(alerts):,}")

extract: 46,793 rows · accounts_d: 12,000 · transactions_d: 46,793 · alerts: 2,618


## Block 1 · `GROUP BY` → `groupby().agg()` (Q1–10)

Named aggregation: `.agg(new_name=("column", "function"))`. Before each table, say what one row of the result is.

### Q1 · EASY · SQL twin: d3a Q6

Number of transactions per `channel_clean`, most first, including missing channels.

💡 *Hint:* `df.groupby("channel_clean", dropna=False).size()`.

In [6]:
# Your code here
df.groupby("channel_clean", dropna=False).size().sort_values(ascending=False)

,0
channel_clean,
Online,8502
Branch,8477
PESONet,8014
Mobile,7233
InstaPay,7228
ATM,7199
NaN,140


### Q2 · EASY · SQL twin: d3a Q7–Q8

In one table: number of transactions, total amount and average amount per `channel_clean`, sorted by total amount.

💡 *Hint:* `.agg(transactions=("tx_id", "count"), total_amount=("amount", "sum"), avg_amount=("amount", "mean"))`.

In [7]:
df.groupby("channel_clean", dropna=False).agg(
    transactions=("tx_id", "count"),
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean")
).sort_values("total_amount", ascending=False)

,transactions,total_amount,avg_amount
channel_clean,,,
Online,8502,"995,848,023.41","117,131.03"
Branch,8477,"920,921,013.62","108,637.61"
PESONet,8014,"799,428,999.89","99,754.06"
ATM,7199,"199,254,148.99","27,678.03"
InstaPay,7228,"176,497,642.19","24,418.60"
Mobile,7233,"150,266,284.59","20,775.10"
NaN,140,"8,326,595.63","59,475.68"


### Q3 · EASY · SQL twin: d3a Q10

Number of accounts per `acct_type`, using the de-duplicated `accounts_d`.

In [8]:
# Your code here
accounts_d.groupby("acct_type").size()

,0
acct_type,
Business,1996
Checking,3792
Payroll,2696
Savings,3516


### Q4 · AVERAGE · SQL twin: d3a Q11

Earliest and latest `open_date` per account type. Convert `open_date` to datetimes first, without changing `accounts_d` itself.

💡 *Hint:* `accounts_d.assign(open_date=pd.to_datetime(accounts_d["open_date"]))` gives a converted copy.

In [9]:
# Your code here
accounts_d.assign(open_date=pd.to_datetime(accounts_d["open_date"], format="mixed")).groupby("acct_type").agg(earliest=("open_date", "min"), latest=("open_date", "max"))

,earliest,latest
acct_type,,
Business,2015-03-05,2025-11-30
Checking,2015-02-28,2025-11-30
Payroll,2015-04-22,2025-11-30
Savings,2015-03-29,2025-11-30


### Q5 · AVERAGE · SQL twin: d3a Q13

For each alert rule: number of alerts, number of **scored** alerts, and the average, minimum and maximum `alert_score_num`.

In [10]:
# Your code here
alerts.groupby("rule_name").agg(alert_count=("tx_id", "count"), scored_alert_count=("alert_score_num", "count"), avg_score=("alert_score_num", "mean"), min_score=("alert_score_num", "min"), max_score=("alert_score_num", "max"))

,alert_count,scored_alert_count,avg_score,min_score,max_score
rule_name,,,,,
Dormant_Reactivation,126,116,62.76,34.80,95.60
High_Velocity_Count,462,446,60.58,16.60,92.90
Large_Amount_CTR,1540,1441,54.94,16.70,95.40
Near_Threshold_Structuring,366,341,72.04,39.40,100.00
Rapid_Movement,124,117,76.28,42.30,100.00


✍️ **Why is "number of alerts" different from "number of scored alerts"?**

*Your answer:* There are alerts that are labeled "NOT SCORED" under the alert_score_num column

### Q6 · AVERAGE · SQL twin: d3a Q25, Q27

For each channel: total transactions, transactions of PHP 500,000 or more, and that share as a percentage, highest share first.

💡 *Hint:* The mean of a 0/1 flag is a rate: `large_rate=("large_amount_flag", "mean")`.

In [11]:
# Your code here
df.groupby("channel_clean", dropna=False).agg(
    total_transactions=("tx_id", "count"),
    large_transactions=("amount", lambda s: (s >= 500000).sum()),
    large_rate=("amount", lambda s: (s >= 500000).mean() * 100)
).sort_values("large_rate", ascending=False)

,total_transactions,large_transactions,large_rate
channel_clean,,,
PESONet,8014,502,6.26
Online,8502,484,5.69
Branch,8477,472,5.57
NaN,140,3,2.14
InstaPay,7228,41,0.57
Mobile,7233,25,0.35
ATM,7199,13,0.18


### Q7 · AVERAGE · SQL twin: d3a Q17

The five channels with the highest total amount.

In [12]:
# Calculate total transaction amount per channel and display the top 5
top_five_channels = df.groupby("channel_clean", dropna=False).agg(
    total_amount=("amount", "sum")
).sort_values("total_amount", ascending=False).head(5)

display(top_five_channels)

,total_amount
channel_clean,
Online,"995,848,023.41"
Branch,"920,921,013.62"
PESONet,"799,428,999.89"
ATM,"199,254,148.99"
InstaPay,"176,497,642.19"


### Q8 · AVERAGE · SQL twin: d3a Q32

For each customer in `accounts_d`: number of accounts, number of distinct account types, earliest and latest opening date. What is the most accounts any one customer holds, and how many customers hold that many?

In [13]:
# Pre-convert open_date to datetime to allow min/max operations
accounts_temp = accounts_d.assign(open_date=pd.to_datetime(accounts_d["open_date"], format="mixed"))

# Group by customer and calculate aggregations
cust_accounts = accounts_temp.groupby("cust_id").agg(
    num_accounts=("acct_id", "count"),
    unique_acct_types=("acct_type", "nunique"),
    earliest_open=("open_date", "min"),
    latest_open=("open_date", "max")
)

# Find the maximum accounts and how many customers hold that many
max_accts = cust_accounts["num_accounts"].max()
num_cust_with_max = (cust_accounts["num_accounts"] == max_accts).sum()

print(f"Maximum accounts held by any one customer: {max_accts}")
print(f"Number of customers holding {max_accts} accounts: {num_cust_with_max}\n")
display(cust_accounts.head())

Maximum accounts held by any one customer: 6
Number of customers holding 6 accounts: 13



,num_accounts,unique_acct_types,earliest_open,latest_open
cust_id,,,,
CUST-0006932d,1,1,2022-08-18,2022-08-18
CUST-00085b82,1,1,2024-09-17,2024-09-17
CUST-000a2af6,1,1,2020-01-17,2020-01-17
CUST-000cf71a,1,1,2020-09-08,2020-09-08
CUST-000fed8e,1,1,2017-02-11,2017-02-11


### Q9 · AVERAGE

Alert rate by originating account status (`orig_acct_status`): transactions, alerted transactions, and alert rate as a percentage. Which status stands out?

In [14]:
# Calculate transaction metrics grouped by originating account status
alert_rate_by_status = df.groupby("orig_acct_status", dropna=False).agg(
    transactions=("tx_id", "count"),
    alerted_transactions=("was_alerted", "sum"),
    alert_rate=("was_alerted", lambda s: s.mean() * 100)
).sort_values("alert_rate", ascending=False)

display(alert_rate_by_status)

,transactions,alerted_transactions,alert_rate
orig_acct_status,,,
Dormant,743,122,16.42
Closed,148,11,7.43
Active,45902,2248,4.90


### Q10 · DIFFICULT

Build the **account behaviour table**: one row per `orig_acct` with `tx_count`, `total_amount`, `max_amount`, `near_threshold_tx`, `active_days` (distinct `tx_date`), `first_tx`, `last_tx` and `external_share` (share of transactions to `EXT-` accounts).

Prove that `orig_acct` is unique in your table and that `total_amount` adds back to the extract total. How many accounts made **3 or more** near-threshold transactions?

💡 *Hint:* `external_share`: make a 0/1 column first (`1 - dest_is_internal`) and take its mean. For float totals, check `abs(a - b) < 0.01` rather than `==`.

In [15]:
import numpy as np
import pandas as pd

# 1. Identify the correct column for near-threshold transactions
threshold_cols = [c for c in df.columns if 'threshold' in c.lower()]
print("Identified threshold-related columns:", threshold_cols)
# We'll use the first matched column, typically 'near_threshold_tx' or 'is_near_threshold'
near_thresh_col = threshold_cols[0] if threshold_cols else None

# 2. Create the external_share helper column
# 1 - dest_is_internal. We can check if dest_acct starts with 'EXT-'
df['dest_is_external'] = df['dest_acct'].str.startswith('EXT-').astype(int)

# 3. Build the account behaviour table
abt = df.groupby('orig_acct').agg(
    tx_count=('tx_id', 'count'),
    total_amount=('amount', 'sum'),
    max_amount=('amount', 'max'),
    near_threshold_tx=(near_thresh_col, 'sum') if near_thresh_col else ('tx_id', lambda x: 0),
    active_days=('tx_date', 'nunique'),
    first_tx=('tx_datetime', 'min'),
    last_tx=('tx_datetime', 'max'),
    external_share=('dest_is_external', 'mean')
).reset_index()

# 4. Prove that orig_acct is unique in this table
is_unique = abt['orig_acct'].is_unique
print(f"Is orig_acct unique in the behaviour table? {is_unique} ({len(abt):,} rows)")

# 5. Prove that total_amount adds back to the extract total
extract_total = df['amount'].sum()
abt_total = abt['total_amount'].sum()
difference = abs(extract_total - abt_total)
is_reconciled = difference < 0.01
print(f"Extract Total:  Php {extract_total:,.2f}")
print(f"ABT Total:      Php {abt_total:,.2f}")
print(f"Reconciles?     {is_reconciled} (Difference: Php {difference:,.4f})")

# 6. How many accounts made 3 or more near-threshold transactions?
high_near_threshold_accts = (abt['near_threshold_tx'] >= 3).sum()
print(f"Accounts with >= 3 near-threshold transactions: {high_near_threshold_accts}")

print("\nPreview of the Account Behaviour Table:")
display(abt.head())

Identified threshold-related columns: ['near_threshold_flag', 'rule_near_threshold']
Is orig_acct unique in the behaviour table? True (11,306 rows)
Extract Total:  Php 3,250,542,708.32
ABT Total:      Php 3,250,542,708.32
Reconciles?     True (Difference: Php 0.0000)
Accounts with >= 3 near-threshold transactions: 74

Preview of the Account Behaviour Table:


,orig_acct,tx_count,total_amount,max_amount,near_threshold_tx,active_days,first_tx,last_tx,external_share
0,ACC-0016130c,1,"1,220.88","1,220.88",0,1,2025-12-16 04:42:32+00:00,2025-12-16 04:42:32+00:00,1.00
1,ACC-001afb6d,2,"2,979.27","2,058.43",0,2,2025-03-25 21:14:00+00:00,2025-07-21 14:49:26+00:00,1.00
2,ACC-00252c05,2,"5,457.60","4,841.95",0,2,2025-03-03 23:21:03+00:00,2025-10-12 21:09:42+00:00,1.00
3,ACC-002a7188,5,"29,405.05","9,816.88",0,5,2025-08-30 19:50:00+00:00,2025-12-15 16:51:14+00:00,0.80
4,ACC-002d18c2,5,"10,613.77","5,756.13",0,5,2025-01-16 02:56:25+00:00,2025-11-20 04:05:38+00:00,1.00


## Block 2 · `JOIN` → `merge()` (Q11–25)

Every merge gets three habits: rows before and after, `validate=` with the expected cardinality, and `indicator=True` when you need to know what didn't match.

### Q11 · EASY · SQL twin: d3b Q1

Merge `accounts_d` with `customers` to show each account's `acct_id`, `acct_type`, `cust_id` and the customer's `age`. Print the row count before and after. Which `how=` did you choose, and why?

In [16]:
# Q11: Merge accounts_d with customers
print(f"Row count in accounts_d (before): {len(accounts_d):,}")
print(f"Row count in customers (before): {len(customers):,}")

# Performing an inner merge since every account should ideally link to a valid customer,
# and we want to prevent any potential mismatch/unmatched accounts.
merged_df = pd.merge(
    accounts_d,
    customers[["cust_id", "age"]],
    on="cust_id",
    how="inner",
    validate="many_to_one"
)

print(f"Row count after inner merge: {len(merged_df):,}")
print("\nPreview of the merged DataFrame:")
display(merged_df[["acct_id", "acct_type", "cust_id", "age"]].head())

Row count in accounts_d (before): 12,000
Row count in customers (before): 10,000
Row count after inner merge: 11,984

Preview of the merged DataFrame:


,acct_id,acct_type,cust_id,age
0,ACC-84237821,Checking,CUST-2fc59321,42
1,ACC-731d07c6,Payroll,CUST-010b5ec1,29
2,ACC-373a10e4,Business,CUST-7d184195,60
3,ACC-9d590a10,Business,CUST-87b646de,46
4,ACC-31b1323d,Business,CUST-3a28ae55,67


### Q12 · AVERAGE

Merge `transactions_d` with the **raw** `accounts` (duplicates included) on `orig_acct` = `acct_id`, using `validate="many_to_one"`. What happens? Catch the error with `try/except` and print it. Then do the same merge with `accounts_d`.

💡 *Hint:* `except pd.errors.MergeError as err: print(err)`.

In [17]:
print('--- Attempting validation of many-to-one on raw accounts (contains duplicate keys) ---')
try:
    # This will fail because raw accounts has duplicates on 'acct_id'
    fail_merge = pd.merge(
        transactions_d,
        accounts,
        left_on='orig_acct',
        right_on='acct_id',
        how='left',
        validate='many_to_one'
    )
except pd.errors.MergeError as err:
    print(f'\nCaught expected MergeError:\n{err}')

print('\n--- Attempting validation of many-to-one on de-duplicated accounts_d ---')
try:
    success_merge = pd.merge(
        transactions_d,
        accounts_d,
        left_on='orig_acct',
        right_on='acct_id',
        how='left',
        validate='many_to_one'
    )
    print(f'Success! Merged table contains {len(success_merge):,} rows.')
    display(success_merge.head())
except Exception as err:
    print(f'An unexpected error occurred: {err}')

--- Attempting validation of many-to-one on raw accounts (contains duplicate keys) ---

Caught expected MergeError:
Merge keys are not unique in right dataset; not a many-to-one merge

--- Attempting validation of many-to-one on de-duplicated accounts_d ---
Success! Merged table contains 46,793 rows.


,tx_id,orig_acct,dest_acct,amount,tx_datetime,channel,tx_type,acct_id,cust_id,acct_type,open_date,initial_balance,acct_status
0,TX-fd9da9bfe0,ACC-98c10d9b,EXT-4a87ad29,"2,931.08",2025-01-01 00:42:54,Online,Outbound,ACC-98c10d9b,CUST-6e0228e0,Savings,2024-04-26,"48,446.59",Active
1,TX-8811a82e45,ACC-431fa4b8,ACC-28aecff7,275.29,2025-01-01 00:46:34,Mobile,Transfer,ACC-431fa4b8,CUST-e135fa10,Checking,2017-06-11,"195,211.08",Active
2,TX-8562cc2b00,ACC-9e050295,EXT-9491368a,676.18,2025-01-01 01:08:07,Mobile,Outbound,ACC-9e050295,CUST-3124e6bb,Business,2023-11-24,"26,641.50",Active
3,TX-9d3d73b735,ACC-60770c65,ACC-1368c508,544.39,2025-01-01 01:08:36,ATM,Transfer,ACC-60770c65,CUST-4127b9ab,Business,2020-10-20,"57,470.17",Active
4,TX-40876d75d0,ACC-9529962c,EXT-420a381d,261.38,2025-01-01 01:12:30,ATM,Outbound,ACC-9529962c,CUST-1f8a7eb1,Savings,2020-06-08,"48,197.26",Active


### Q13 · AVERAGE · SQL twin: d3b Q6

Every customer and any accounts they hold, including customers with none. Use `indicator=True`. How many customers have no accounts?

In [18]:
# Q13: Merge customers with accounts_d using indicator=True
cust_acct_merge = pd.merge(
    customers,
    accounts_d,
    on="cust_id",
    how="left",
    indicator=True,
    validate="one_to_many"
)

# Count customers with no accounts (indicated by 'left_only' in the _merge column)
no_acct_customers = cust_acct_merge[cust_acct_merge["_merge"] == "left_only"]
num_no_acct = no_acct_customers["cust_id"].nunique()

print(f"Total unique customers in master table: {customers['cust_id'].nunique():,}")
print(f"Total rows after left-merge: {len(cust_acct_merge):,}")
print(f"Number of customers with NO accounts: {num_no_acct:,}")

if num_no_acct > 0:
    print("\nPreview of customers with no accounts:")
    display(no_acct_customers[["cust_id", "occupation", "declared_income", "_merge"]].head())

Total unique customers in master table: 10,000
Total rows after left-merge: 11,997
Number of customers with NO accounts: 13

Preview of customers with no accounts:


,cust_id,occupation,declared_income,_merge
1108,CUST-4ad33f7c,Business Owner,"2,214,083.97",left_only
3169,CUST-dedf0c76,Business Owner,"2,019,705.41",left_only
4286,CUST-4bec12e1,Business Owner,"1,871,280.60",left_only
5240,CUST-7f7675df,Professional,NaN,left_only
5671,CUST-ff1212f2,Professional,"1,156,057.12",left_only


### Q14 · AVERAGE · SQL twin: d3b Q9

Accounts in `accounts_d` that have **never received** a transaction (in `transactions_d`). How many?

💡 *Hint:* Left-merge with the distinct `dest_acct` values and keep `_merge == "left_only"`, or use `~accounts_d["acct_id"].isin(...)`.

In [ ]:
# Q14: Accounts in accounts_d that have never received a transaction
never_received = accounts_d[~accounts_d["acct_id"].isin(transactions_d["dest_acct"])]
num_never_received = len(never_received)

print(f"Total unique accounts in accounts_d: {len(accounts_d):,}")
print(f"Number of accounts that have never received a transaction: {num_never_received:,}")

if num_never_received > 0:
    print("\nPreview of accounts that never received a transaction:")
    display(never_received.head())

Total unique accounts in accounts_d: 12,000
Number of accounts that have never received a transaction: 3,799

Preview of accounts that never received a transaction:


,acct_id,cust_id,acct_type,open_date,initial_balance,acct_status
2,ACC-373a10e4,CUST-7d184195,Business,2025-11-30,"27,606.87",Active
3,ACC-9d590a10,CUST-87b646de,Business,2020-04-25,"69,816.69",Active
4,ACC-31b1323d,CUST-3a28ae55,Business,2020-01-02,"11,091.49",Dormant
6,ACC-36a123f5,CUST-909fdfb3,Savings,2016-12-26,"473,995.39",Active
11,ACC-3c55bef3,CUST-66b181ae,Business,2025-01-30,"34,473.80",Active


### Q15 · AVERAGE · SQL twin: d3b Q10

Transactions in `transactions_d` that generated **no** alerts. How many?

In [19]:
# Q15: Transactions in transactions_d that generated no alerts
tx_alerts_merge = pd.merge(
    transactions_d,
    alerts[["tx_id", "alert_id"]],
    on="tx_id",
    how="left",
    indicator=True
)

# Count transactions with no matching alerts (labeled 'left_only')
no_alert_txs = tx_alerts_merge[tx_alerts_merge["_merge"] == "left_only"]
num_no_alerts = len(no_alert_txs)

print(f"Total transactions in transactions_d: {len(transactions_d):,}")
print(f"Number of transactions that generated NO alerts: {num_no_alerts:,}")
print(f"Percentage of alert-free transactions: {num_no_alerts / len(transactions_d) * 100:.2f}%")

if num_no_alerts > 0:
    print(
        "\nPreview of transactions with no alerts:"
    )
    display(no_alert_txs.head())

Total transactions in transactions_d: 46,793
Number of transactions that generated NO alerts: 44,412
Percentage of alert-free transactions: 94.91%

Preview of transactions with no alerts:


,tx_id,orig_acct,dest_acct,amount,tx_datetime,channel,tx_type,alert_id,_merge
0,TX-fd9da9bfe0,ACC-98c10d9b,EXT-4a87ad29,"2,931.08",2025-01-01 00:42:54,Online,Outbound,NaN,left_only
1,TX-8811a82e45,ACC-431fa4b8,ACC-28aecff7,275.29,2025-01-01 00:46:34,Mobile,Transfer,NaN,left_only
2,TX-8562cc2b00,ACC-9e050295,EXT-9491368a,676.18,2025-01-01 01:08:07,Mobile,Outbound,NaN,left_only
3,TX-9d3d73b735,ACC-60770c65,ACC-1368c508,544.39,2025-01-01 01:08:36,ATM,Transfer,NaN,left_only
4,TX-40876d75d0,ACC-9529962c,EXT-420a381d,261.38,2025-01-01 01:12:30,ATM,Outbound,NaN,left_only


### Q16 · AVERAGE · SQL twin: Day 4 inflation demo

Left-merge `alerts` onto `transactions_d` on `tx_id`. How many rows do you get, and what happens to the amount total?

In [20]:
# Q16: Left-merge alerts onto transactions_d
tx_alerts_left = pd.merge(
    transactions_d,
    alerts[["tx_id", "alert_id"]],
    on="tx_id",
    how="left"
)

original_rows = len(transactions_d)
merged_rows = len(tx_alerts_left)
original_amount = transactions_d["amount"].sum()
merged_amount = tx_alerts_left["amount"].sum()

print(f"Original transactions_d: {original_rows:,} rows, Total Amount: Php {original_amount:,.2f}")
print(f"Merged DataFrame:        {merged_rows:,} rows, Total Amount: Php {merged_amount:,.2f}")
print(f"Difference:              {merged_rows - original_rows:,} extra rows, Php {merged_amount - original_amount:,.2f} extra money")

Original transactions_d: 46,793 rows, Total Amount: Php 3,250,542,708.32
Merged DataFrame:        47,030 rows, Total Amount: Php 3,447,731,181.43
Difference:              237 extra rows, Php 197,188,473.11 extra money


✍️ **Explain the extra rows and the extra money in one sentence.**

*Your answer:* The extra rows and inflated transaction amount are caused by a one-to-many relationship where a single transaction triggers multiple alerts, thereby duplicating that transaction row and its corresponding amount during the merge.

In [21]:
# Q17: Aggregate alerts to one row per tx_id first, then left-merge
alerts_agg = alerts.groupby("tx_id").agg(
    alert_count=("rule_name", "count"),
    rules_fired=("rule_name", lambda s: " | ".join(sorted(s.unique())))
).reset_index()

# Left-merge with validation "one_to_one"
tx_alerts_fixed = pd.merge(
    transactions_d,
    alerts_agg,
    on="tx_id",
    how="left",
    validate="one_to_one"
)

# Fill NaN values for transactions that didn't generate any alerts
tx_alerts_fixed["alert_count"] = tx_alerts_fixed["alert_count"].fillna(0).astype(int)
tx_alerts_fixed["rules_fired"] = tx_alerts_fixed["rules_fired"].fillna("")

# Verification metrics
original_rows = len(transactions_d)
fixed_rows = len(tx_alerts_fixed)
original_amount = transactions_d["amount"].sum()
fixed_amount = tx_alerts_fixed["amount"].sum()
total_alerts_mapped = tx_alerts_fixed["alert_count"].sum()

print(f"Row Count Reconciles?          {original_rows == fixed_rows} ({fixed_rows:,} vs {original_rows:,})")
print(f"Amount Reconciles?              {abs(original_amount - fixed_amount) < 0.01} (Php {fixed_amount:,.2f} vs Php {original_amount:,.2f})")
print(f"Alerts Mapped Total Reconciles? {total_alerts_mapped == 2618} (Sum: {total_alerts_mapped:,} vs expected 2,618)")

print("\nPreview of the fixed merged table:")
display(tx_alerts_fixed[tx_alerts_fixed["alert_count"] > 0].head())

Row Count Reconciles?          True (46,793 vs 46,793)
Amount Reconciles?              True (Php 3,250,542,708.32 vs Php 3,250,542,708.32)
Alerts Mapped Total Reconciles? True (Sum: 2,618 vs expected 2,618)

Preview of the fixed merged table:


,tx_id,orig_acct,dest_acct,amount,tx_datetime,channel,tx_type,alert_count,rules_fired
13,TX-5188c1983b,ACC-888b8eea,EXT-5e9e7e45,"1,488,430.19",2025-01-01 04:35:40,Branch,Business Payment,1,Large_Amount_CTR
44,TX-ab205dd25a,ACC-67131798,EXT-3e3562bb,"1,701,769.20",2025-01-01 14:01:15,PESONet,Business Payment,1,Large_Amount_CTR
80,TX-0be940d6c9,ACC-df97c6e4,EXT-dc48efde,"815,576.67",2025-01-02 01:12:44,Branch,Business Payment,1,Large_Amount_CTR
97,TX-0778f51ae6,ACC-71f54212,ACC-a49414ea,"715,471.15",2025-01-02 08:10:28,PESONet,Business Payment,1,Large_Amount_CTR
106,TX-029d74c9a0,ACC-50fe8098,ACC-63bf9b0b,"565,144.41",2025-01-02 10:28:33,Branch,Business Payment,1,Large_Amount_CTR


### Q17 · DIFFICULT · SQL twin: 04 SQL, alerts_per_tx

Fix Q16 the way the SQL did: aggregate `alerts` to **one row per `tx_id`** first (`alert_count`, and `rules_fired` as text joined with `' | '`), then merge with `validate="one_to_one"`. Prove that rows and the amount total are unchanged, and that alerts still add up to 2,618.

💡 *Hint:* `rules_fired=("rule_name", lambda s: " | ".join(sorted(s.unique())))`. After a left merge, transactions with no alerts have NaN counts: `.fillna(0)`.

In [22]:
# Q17: Aggregate alerts to one row per tx_id first, then left-merge with transactions_d
alerts_agg = alerts.groupby("tx_id").agg(
    alert_count=("rule_name", "count"),
    rules_fired=("rule_name", lambda s: " | ".join(sorted(s.unique())))
).reset_index()

# Perform left merge validating one-to-one mapping
tx_alerts_fixed = pd.merge(
    transactions_d,
    alerts_agg,
    on="tx_id",
    how="left",
    validate="one_to_one"
)

# Fill missing values for transaction alerts
tx_alerts_fixed["alert_count"] = tx_alerts_fixed["alert_count"].fillna(0).astype(int)
tx_alerts_fixed["rules_fired"] = tx_alerts_fixed["rules_fired"].fillna("")

# Verification metrics
original_rows = len(transactions_d)
fixed_rows = len(tx_alerts_fixed)
original_amount = transactions_d["amount"].sum()
fixed_amount = tx_alerts_fixed["amount"].sum()
total_alerts_mapped = tx_alerts_fixed["alert_count"].sum()

print(f"Row Count Reconciles?          {original_rows == fixed_rows} ({fixed_rows:,} vs {original_rows:,})")
print(f"Amount Reconciles?              {abs(original_amount - fixed_amount) < 0.01} (Php {fixed_amount:,.2f} vs Php {original_amount:,.2f})")
print(f"Alerts Mapped Total Reconciles? {total_alerts_mapped == 2618} (Sum: {total_alerts_mapped:,} vs expected 2,618)")

Row Count Reconciles?          True (46,793 vs 46,793)
Amount Reconciles?              True (Php 3,250,542,708.32 vs Php 3,250,542,708.32)
Alerts Mapped Total Reconciles? True (Sum: 2,618 vs expected 2,618)


### Q18 · AVERAGE

Merge `alerts` with `investigations` (one row per alert, `validate="one_to_one"`). Then group alerts into score bands: **Critical** (≥ 90), **High** (≥ 80), **Medium** (≥ 60), **Low**, and **Unscored**. For each band: alerts, investigated, SARs, investigation rate, and SARs per investigated alert.

Is a higher-scored alert more likely to be **investigated**? More likely to end in a **SAR**?

💡 *Hint:* `np.select([score >= 90, score >= 80, score >= 60, score.notna()], ["Critical", "High", "Medium", "Low"], default="Unscored")`. Count investigated with `("case_id", "count")`.

In [23]:
import numpy as np
import pandas as pd

# 1. Inspect the possible values in investigation_result to accurately identify SARs
print("Investigation results in the dataset:")
print(investigations['investigation_result'].value_counts(dropna=False))


Investigation results in the dataset:
investigation_result
No_Action    1043
SAR_Filed     268
Pending       135
Escalated     129
NaN            48
Name: count, dtype: int64


In [24]:
# 2. Merge alerts and investigations (one-to-one validation)
alerts_inv = pd.merge(
    alerts,
    investigations,
    on="alert_id",
    how="left",
    validate="one_to_one"
)

# 3. Create score bands using np.select
score = alerts_inv["alert_score_num"]
conditions = [
    score >= 90,
    score >= 80,
    score >= 60,
    score.notna()
]
choices = ["Critical", "High", "Medium", "Low"]
alerts_inv["score_band"] = np.select(conditions, choices, default="Unscored")

# Define order for categorization
band_order = ["Critical", "High", "Medium", "Low", "Unscored"]
alerts_inv["score_band"] = pd.Categorical(alerts_inv["score_band"], categories=band_order, ordered=True)

# Define SAR definition based on investigation results (typically 'SAR' or 'Escalated_to_SAR')
sar_mask = alerts_inv["investigation_result"].str.upper().str.contains("SAR", na=False)
alerts_inv["is_sar"] = sar_mask.astype(int)

# Determine if an alert was investigated (has a valid case_id)
alerts_inv["is_investigated"] = alerts_inv["case_id"].notna().astype(int)

# 4. Group by score band and aggregate metrics
band_summary = alerts_inv.groupby("score_band", observed=False).agg(
    alerts=("alert_id", "count"),
    investigated=("is_investigated", "sum"),
    sars=("is_sar", "sum")
).reset_index()

# Calculate rates
band_summary["investigation_rate_pct"] = (band_summary["investigated"] / band_summary["alerts"]) * 100
band_summary["sars_per_investigated_pct"] = np.where(
    band_summary["investigated"] > 0,
    (band_summary["sars"] / band_summary["investigated"]) * 100,
    0.0
)

display(band_summary)


,score_band,alerts,investigated,sars,investigation_rate_pct,sars_per_investigated_pct
0,Critical,50,28,11,56.00,39.29
1,High,148,92,26,62.16,28.26
2,Medium,985,603,113,61.22,18.74
3,Low,1278,798,102,62.44,12.78
4,Unscored,157,102,16,64.97,15.69


In [ ]:
# Your code here


In [25]:
# Filter transactions of PHP 500,000 or more
large_tx = transactions_d[transactions_d["amount"] >= 500000].copy()

# Left-merge with accounts_d to retrieve the destination account type
# Since some destinations are external (starting with 'EXT-'), they won't match keys in accounts_d
large_tx_merged = pd.merge(
    large_tx,
    accounts_d[["acct_id", "acct_type"]].rename(columns={"acct_type": "dest_acct_type"}),
    left_on="dest_acct",
    right_on="acct_id",
    how="left",
    validate="many_to_one"
)

# Classify destination as 'Internal' if there is a match in accounts_d, or 'External' if it starts with 'EXT-'
# Otherwise mark as 'Unknown'
conditions = [
    large_tx_merged["dest_acct_type"].notna(),
    large_tx_merged["dest_acct"].str.startswith("EXT-", na=False)
]
choices = ["Internal", "External"]
large_tx_merged["destination_status"] = np.select(conditions, choices, default="Unknown")

# Group and count the destinations
destination_summary = large_tx_merged.groupby("destination_status", dropna=False).agg(
    transactions=("tx_id", "count"),
    total_amount=("amount", "sum")
).reset_index()

display(destination_summary)


,destination_status,transactions,total_amount
0,External,987,"1,168,596,521.17"
1,Internal,553,"655,150,311.66"


✍️ **What does this table suggest about how investigators choose which alerts to work?**

*Your answer:* The table suggests that investigators do **not** prioritize alerts based on their risk score, since the investigation rate remains remarkably flat (ranging between 56% to 65%) across all score bands (including Unscored). However, the rising SAR yield per investigated alert (from 12.78% in Low to 39.29% in Critical) highlights that investigators could be significantly more productive and file more SARs if they prioritized higher-scoring alerts first.

### Q19 · AVERAGE · SQL twin: d3b Q5

For transactions in `transactions_d` of PHP 500,000 or more, attach the **destination** account type by merging `dest_acct` to `accounts_d`. How many stay inside the bank, and how many leave it?

In [26]:
# Q19: Attach destination account type and classify as Internal vs External
large_tx = transactions_d[transactions_d["amount"] >= 500000].copy()

# Left-merge with accounts_d to retrieve the destination account type
large_tx_merged = pd.merge(
    large_tx,
    accounts_d[["acct_id", "acct_type"]].rename(columns={"acct_type": "dest_acct_type"}),
    left_on="dest_acct",
    right_on="acct_id",
    how="left",
    validate="many_to_one"
)

# Classify destination as 'Internal' if there is a match in accounts_d, or 'External' if it starts with 'EXT-'
conditions = [
    large_tx_merged["dest_acct_type"].notna(),
    large_tx_merged["dest_acct"].str.startswith("EXT-", na=False)
]
choices = ["Internal", "External"]
large_tx_merged["destination_status"] = np.select(conditions, choices, default="Unknown")

# Group and count the destinations
destination_summary = large_tx_merged.groupby("destination_status", dropna=False).agg(
    transactions=("tx_id", "count"),
    total_amount=("amount", "sum")
).reset_index()

display(destination_summary)


,destination_status,transactions,total_amount
0,External,987,"1,168,596,521.17"
1,Internal,553,"655,150,311.66"


### Q20 · AVERAGE · SQL twin: d3b Q11–Q12

Received side: for transactions that land on an internal account, show per destination `acct_type` the number of transactions, the total received amount, and the number of distinct receiving accounts.

In [27]:
# Q20: Merge all transactions in transactions_d with accounts_d on dest_acct to find internal transfers
internal_receives = pd.merge(
    transactions_d,
    accounts_d[["acct_id", "acct_type"]].rename(columns={"acct_type": "dest_acct_type"}),
    left_on="dest_acct",
    right_on="acct_id",
    how="inner",  # Inner merge ensures we only capture internal accounts present in accounts_d
    validate="many_to_one"
)

# Group by destination account type and aggregate metrics
received_summary = internal_receives.groupby("dest_acct_type").agg(
    transactions=("tx_id", "count"),
    total_received_amount=("amount", "sum"),
    distinct_receiving_accounts=("dest_acct", "nunique")
).reset_index()

display(received_summary)


,dest_acct_type,transactions,total_received_amount,distinct_receiving_accounts
0,Business,2325,"193,142,962.09",1385
1,Checking,4676,"440,995,741.67",2618
2,Payroll,3084,"287,725,532.47",1840
3,Savings,4237,"405,160,087.37",2358


### Q21 · AVERAGE

Build the **customer table**: one row per `orig_cust_id` from `df`, with `tx_count`, `total_sent`, `alerts` (sum of `alert_count`) and `sars` (sum of `sar_count`). Then merge `customers` for `occupation`, `declared_income` and `kyc_risk_score`, with `validate="one_to_one"` and `indicator=True`. Call it `cust`.

In [ ]:
# Your code here


In [28]:
# Q21: Build the customer table (cust)
# 1. Group the main analytical extract (df) by 'orig_cust_id' and aggregate metrics
cust_agg = df.groupby("orig_cust_id").agg(
    tx_count=("tx_id", "count"),
    total_sent=("amount", "sum"),
    alerts=("alert_count", "sum"),
    sars=("sar_count", "sum")
).reset_index()

# 2. Merge with 'customers' to bring in demographic and risk fields
cust = pd.merge(
    cust_agg,
    customers[["cust_id", "occupation", "declared_income", "kyc_risk_score"]],
    left_on="orig_cust_id",
    right_on="cust_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

print(f"Customer table (cust) built with {len(cust):,} rows.")
display(cust.head())

Customer table (cust) built with 9,499 rows.


,orig_cust_id,tx_count,total_sent,alerts,sars,cust_id,occupation,declared_income,kyc_risk_score,_merge
0,CUST-0006932d,3,"6,280.35",0,0,CUST-0006932d,NaN,"96,185.12",44.00,both
1,CUST-00085b82,7,"6,392.48",0,0,CUST-00085b82,Professional,"1,909,061.25",47.00,both
2,CUST-000a2af6,4,"13,108.42",0,0,CUST-000a2af6,Business Owner,"890,725.03",36.00,both
3,CUST-000cf71a,3,"5,766.02",0,0,CUST-000cf71a,Business Owner,"372,387.72",23.00,both
4,CUST-000fed8e,3,863.15,0,0,CUST-000fed8e,business owner,"2,004,115.09",26.00,both


### Q22 · AVERAGE

Reconcile `cust`: does `total_sent` add up to the extract total? How many customers in `cust` have **no** record in `customers`?

In [ ]:
# Your code here


In [29]:
# Q22: Reconcile cust table

# 1. Does total_sent add up to the extract total?
extract_total = df["amount"].sum()
cust_total = cust["total_sent"].sum()
diff = abs(extract_total - cust_total)
is_reconciled = diff < 0.01

print(f"Extract total amount:  Php {extract_total:,.2f}")
print(f"Customer total_sent:   Php {cust_total:,.2f}")
print(f"Do they reconcile?     {is_reconciled} (Difference: Php {diff:,.4f})")

# 2. How many customers in cust have no record in customers?
# The '_merge' column from our left merge tells us if the record was only in the left table ('left_only')
missing_customers_count = (cust["_merge"] == "left_only").sum()
print(f"Customers in cust with no record in customers: {missing_customers_count}")

if missing_customers_count > 0:
    display(cust[cust["_merge"] == "left_only"].head())

Extract total amount:  Php 3,250,542,708.32
Customer total_sent:   Php 3,250,542,708.32
Do they reconcile?     True (Difference: Php 0.0000)
Customers in cust with no record in customers: 14


,orig_cust_id,tx_count,total_sent,alerts,sars,cust_id,occupation,declared_income,kyc_risk_score,_merge
167,CUST-04255755,1,"2,579.85",0,0,NaN,NaN,NaN,NaN,left_only
1234,CUST-21cd088a,3,"3,641.69",0,0,NaN,NaN,NaN,NaN,left_only
2441,CUST-421db101,6,"7,221.85",0,0,NaN,NaN,NaN,NaN,left_only
2958,CUST-4f9f88dd,3,"8,052.00",0,0,NaN,NaN,NaN,NaN,left_only
4119,CUST-6eb9fa75,6,"1,209,110.13",1,0,NaN,NaN,NaN,NaN,left_only


### Q23 · AVERAGE

Add `value_to_income` = `total_sent` ÷ `declared_income`, NaN when income is 0 or missing. How many customers sent **more than** their declared annual income? More than **5×**? Show the top 10.

In [ ]:
# Your code here


In [30]:
# Q23: Add value_to_income and analyze extreme senders

# Calculate value_to_income ratio, setting it to NaN if declared_income is 0 or missing
cust["value_to_income"] = np.where(
    (cust["declared_income"] > 0) & (cust["declared_income"].notna()),
    cust["total_sent"] / cust["declared_income"],
    np.nan
)

# Count customers sending more than their declared annual income (> 1x)
sent_above_income = (cust["value_to_income"] > 1).sum()

# Count customers sending more than 5x their declared annual income (> 5x)
sent_above_5x_income = (cust["value_to_income"] > 5).sum()

print(f"Customers who sent more than their declared annual income (> 1x): {sent_above_income:,}")
print(f"Customers who sent more than 5x their declared annual income (> 5x): {sent_above_5x_income:,}\n")

# Display the top 10 customers by value_to_income
print("Top 10 customers by value-to-income ratio:")
top_10_senders = cust.sort_values("value_to_income", ascending=False).head(10)
display(top_10_senders[["orig_cust_id", "occupation", "declared_income", "total_sent", "value_to_income", "alerts", "sars"]])

Customers who sent more than their declared annual income (> 1x): 485
Customers who sent more than 5x their declared annual income (> 5x): 233

Top 10 customers by value-to-income ratio:


,orig_cust_id,occupation,declared_income,total_sent,value_to_income,alerts,sars
5065,CUST-88b6407a,NaN,"36,851.29","16,213,943.37",439.98,24,7
2912,CUST-4e8d5649,unemployed,"36,625.92","14,243,391.39",388.89,24,9
4132,CUST-6f18980f,Unemployed,"34,828.46","12,774,301.27",366.78,10,5
366,CUST-09ab7916,Student,"27,048.35","9,699,726.32",358.61,16,7
3996,CUST-6b21c179,Student,"57,054.40","16,250,382.39",284.82,23,8
9287,CUST-faca7d6b,Student,"58,915.90","15,000,844.32",254.61,23,10
2934,CUST-4f128eeb,Unemployed,"24,883.96","6,182,110.08",248.44,7,0
5291,CUST-8f9db7ac,Unemployed,"70,081.34","17,301,540.28",246.88,22,6
268,CUST-06e910d0,Unemployed,"34,341.75","6,693,379.88",194.91,7,0
8648,CUST-ea0d58d1,Student,"34,844.53","6,412,244.77",184.02,10,3


### Q24 · DIFFICULT

Compare customers who sent more than their declared income with everyone else: number of customers, total SARs, and each group's share of all SARs.

💡 *Hint:* `np.where(cust["value_to_income"] > 1, "sent > declared income", "rest (incl. unknown)")`, then `groupby`.

In [ ]:
# Your code here


In [31]:
# Q24: Compare extreme senders with everyone else

# Create the group classification
cust["sender_group"] = np.where(
    cust["value_to_income"] > 1,
    "sent > declared income",
    "rest (incl. unknown)"
)

# Aggregate number of customers and total SARs
sar_comparison = cust.groupby("sender_group").agg(
    customers=("orig_cust_id", "count"),
    total_sars=("sars", "sum")
).reset_index()

# Calculate each group's share of all SARs
total_sars_all = sar_comparison["total_sars"].sum()
sar_comparison["sar_share_pct"] = (sar_comparison["total_sars"] / total_sars_all) * 100

# Display the comparison table
display(sar_comparison)

,sender_group,customers,total_sars,sar_share_pct
0,rest (incl. unknown),9014,16,5.97
1,sent > declared income,485,252,94.03


✍️ **What does this suggest, and what can't it prove?**

Extremely high risk profile: Customers who send more than their declared annual income are highly correlated with suspicious behavior. While they make up less than 5% of the active customer base, they account for 94% of all SAR filings. This suggests that the value-to-income metric is an exceptionally strong risk indicator and should be a top priority for transaction monitoring rules and automated alerts.

### Q25 · DIFFICULT · SQL twin: d3b Q19, Q39

Classify each transaction's customer by KYC risk: **High** (≥ 80), **Elevated** (≥ 60), **Standard**, and **Unknown** when the score is missing. Then:
1. per risk category: distinct customers, transactions and alert rate;
2. a pivot table of alert rate (%) with risk categories as rows and `channel_clean` as columns.

💡 *Hint:* Put the missing-score condition first in `np.select`, or missing scores fall into the default. Pivot: `df.pivot_table(index=..., columns=..., values="was_alerted", aggfunc="mean")`.

In [ ]:
# Your code here


In [32]:
# Q25: KYC Risk Classification and Alert Rate Analysis

# 1. Classify each transaction's customer by KYC risk
# We need to map 'kyc_risk_score' to risk categories. Since kyc_risk_score is in customers,
# we can left-merge the kyc_risk_score onto our analytical extract 'df'.
if "kyc_risk_score" not in df.columns:
    # Bring kyc_risk_score from customers into df via orig_cust_id
    df = pd.merge(
        df,
        customers[["cust_id", "kyc_risk_score"]],
        left_on="orig_cust_id",
        right_on="cust_id",
        how="left",
        validate="many_to_one"
    ).drop(columns=["cust_id"])

# Create classification using np.select
score_val = df["kyc_risk_score"]
conditions = [
    score_val.isna(),
    score_val >= 80,
    score_val >= 60
]
choices = ["Unknown", "High", "Elevated"]
df["kyc_risk_category"] = np.select(conditions, choices, default="Standard")

# Define sorting order for risk categories
risk_order = ["High", "Elevated", "Standard", "Unknown"]
df["kyc_risk_category"] = pd.Categorical(df["kyc_risk_category"], categories=risk_order, ordered=True)

# 2. Per risk category: distinct customers, transactions and alert rate
risk_summary = df.groupby("kyc_risk_category", observed=False).agg(
    distinct_customers=("orig_cust_id", "nunique"),
    transactions=("tx_id", "count"),
    alert_rate=("was_alerted", lambda s: s.mean() * 100)
).reset_index()

print("=== Summary by KYC Risk Category ===")
display(risk_summary)

# 3. Pivot table of alert rate (%) with risk categories as rows and channel_clean as columns
pivot_df = df.pivot_table(
    index="kyc_risk_category",
    columns="channel_clean",
    values="was_alerted",
    aggfunc=lambda s: s.mean() * 100,
    observed=False
)

print("\n=== Pivot Table: Alert Rate (%) by Risk Category & Channel ===")
display(pivot_df)

=== Summary by KYC Risk Category ===


,kyc_risk_category,distinct_customers,transactions,alert_rate
0,High,32,175,0.00
1,Elevated,524,2677,4.74
2,Standard,8929,43892,5.13
3,Unknown,14,49,2.04



=== Pivot Table: Alert Rate (%) by Risk Category & Channel ===


channel_clean,ATM,Branch,InstaPay,Mobile,Online,PESONet
kyc_risk_category,,,,,,
High,0.00,0.00,0.00,0.00,0.00,0.00
Elevated,3.54,7.56,1.18,1.46,6.88,7.08
Standard,2.81,8.25,1.73,1.10,8.32,7.25
Unknown,8.33,0.00,0.00,0.00,0.00,0.00


---
### Done?

Unfinished questions are homework. Bring the one you were most stuck on to tomorrow's opener, along with what you tried.